In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# CIFAR-10: 60,000 colour images, 10 classes

transform_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),   # 3 channels — RGB
                         (0.5, 0.5, 0.5))
])

train_cifar = datasets.CIFAR10('./data', train=True,
                                download=True,
                                transform=transform_cifar)
test_cifar  = datasets.CIFAR10('./data', train=False,
                                transform=transform_cifar)

train_loader_c = DataLoader(train_cifar, batch_size=64, shuffle=True)
test_loader_c  = DataLoader(test_cifar,  batch_size=256)

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss   = criterion(output, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct    += (output.argmax(1) == y_batch).sum().item()
        total      += len(y_batch)

    return total_loss / len(loader), correct / total


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            output   = model(X_batch)
            correct += (output.argmax(1) == y_batch).sum().item()
            total   += len(y_batch)
    return correct / total

# CIFAR needs a bigger network — 3 channels, 32x32 images
class CIFARNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)   # 3 channels in
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(128 * 4 * 4, 256)
        self.fc2   = nn.Linear(256, 10)
        self.drop  = nn.Dropout(0.4)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 32x32 → 16x16
        x = self.pool(F.relu(self.conv2(x)))  # 16x16 → 8x8
        x = self.pool(F.relu(self.conv3(x)))  # 8x8   → 4x4
        x = x.view(x.size(0), -1)             # flatten
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        x = self.fc2(x)
        return x

# Train
cifar_model = CIFARNet()
optimizer   = torch.optim.Adam(cifar_model.parameters(), lr=0.001)
criterion   = nn.CrossEntropyLoss()

for epoch in range(20):
    loss, train_acc = train_epoch(cifar_model, train_loader_c,
                                   criterion, optimizer)
    test_acc        = evaluate(cifar_model, test_loader_c)
    print(f"Epoch {epoch+1:2d} | Loss: {loss:.4f} "
          f"| Train: {train_acc:.4f} | Test: {test_acc:.4f}")

100%|██████████| 170M/170M [1:03:35<00:00, 44.7kB/s] 


Epoch  1 | Loss: 1.4770 | Train: 0.4569 | Test: 0.5803
Epoch  2 | Loss: 1.0857 | Train: 0.6120 | Test: 0.6719
Epoch  3 | Loss: 0.9041 | Train: 0.6818 | Test: 0.7063
Epoch  4 | Loss: 0.7900 | Train: 0.7247 | Test: 0.7292
Epoch  5 | Loss: 0.7023 | Train: 0.7528 | Test: 0.7448
Epoch  6 | Loss: 0.6294 | Train: 0.7796 | Test: 0.7637
Epoch  7 | Loss: 0.5633 | Train: 0.8004 | Test: 0.7619
Epoch  8 | Loss: 0.5113 | Train: 0.8215 | Test: 0.7704
Epoch  9 | Loss: 0.4612 | Train: 0.8344 | Test: 0.7642
Epoch 10 | Loss: 0.4173 | Train: 0.8490 | Test: 0.7598
Epoch 11 | Loss: 0.3743 | Train: 0.8653 | Test: 0.7658
Epoch 12 | Loss: 0.3458 | Train: 0.8758 | Test: 0.7750
Epoch 13 | Loss: 0.3165 | Train: 0.8860 | Test: 0.7728
Epoch 14 | Loss: 0.2883 | Train: 0.8956 | Test: 0.7678
Epoch 15 | Loss: 0.2645 | Train: 0.9034 | Test: 0.7701
Epoch 16 | Loss: 0.2455 | Train: 0.9094 | Test: 0.7707
Epoch 17 | Loss: 0.2334 | Train: 0.9160 | Test: 0.7674
Epoch 18 | Loss: 0.2186 | Train: 0.9199 | Test: 0.7659
Epoch 19 |